In [1]:
import os
import re
import io
import time
import json
import requests
import pandas as pd
from urllib.parse import urlparse
from collections import defaultdict, Counter
from pypdf import PdfReader
from docx import Document
from PIL import Image                    # ← FIX: was missing
import pytesseract                       # ← FIX
import fitz                              # ← FIX
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document as LangchainDocument
from openai import OpenAI

# Windows only — set Tesseract path if needed
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

C:\Users\KOUSHIK\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\KOUSHIK\AppData\Local\Temp\ipykernel_3284\2713887965.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
EXCEL_FILE = 'Tracker.xlsx'
CV_FOLDER = 'downloaded_cvs'

In [3]:
# Read Excel WITH header row (default behavior is header=0)
df = pd.read_excel(EXCEL_FILE)

name_map = {}   # filename → {name, email}

for index, row in df.iterrows():
    name  = str(row.iloc[0]).strip()
    email = str(row.iloc[1]).strip()

    if not name or name == "nan":
        continue

    # The downloader used: filename = f"{safe_name}_{index + 2}{ext}"
    # So the filename suffix tells us EXACTLY which row it came from.
    excel_row = index + 2

    # Find the file that ends with _{excel_row}.pdf or _{excel_row}.docx
    for filename in os.listdir(CV_FOLDER):
        # Extract the number before the extension
        match = re.search(r"_(\d+)\.(pdf|docx)$", filename, re.IGNORECASE)
        if not match:
            continue

        file_row = int(match.group(1))

        if file_row == excel_row:
            name_map[filename] = {"name": name, "email": email}
            print(f"✅ {filename}  →  {name}")
            break

print(f"\n✅ Total matched: {len(name_map)}")

✅ Usha.c.t_2.pdf  →  Usha.c.t
✅ banda indu_3.pdf  →  banda indu
✅ t anil_4.pdf  →  t anil
✅ h pavan kalyan_5.pdf  →  h pavan kalyan
✅ kiran_6.pdf  →  kiran
✅ rahul_7.pdf  →  rahul
✅ dinesh kumar manto_8.pdf  →  dinesh kumar manto
✅ mohammed akif ur rahman_9.pdf  →  mohammed akif ur rahman
✅ Dolagobinda Mohanty_10.docx  →  Dolagobinda Mohanty
✅ arpita_11.docx  →  arpita
✅ Deepali Dinesh B._12.pdf  →  Deepali Dinesh B.
✅ r ekambaram_13.pdf  →  r ekambaram
✅ Umarfarooq Pakali_14.pdf  →  Umarfarooq Pakali
✅ deepthi_15.pdf  →  deepthi
✅ m g usharani_16.pdf  →  m g usharani
✅ Sharath Chikkagoudra_17.pdf  →  Sharath Chikkagoudra
✅ Sirishala pavani_18.pdf  →  Sirishala pavani
✅ Pallavi S_19.pdf  →  Pallavi S
✅ Pallavi Gidnandi_20.pdf  →  Pallavi Gidnandi
✅ Abhilash N_21.pdf  →  Abhilash N
✅ Sudharshan Reddy Kuluru_22.pdf  →  Sudharshan Reddy Kuluru
✅ m bhagya lakshmi_23.pdf  →  m bhagya lakshmi
✅ narasimha_24.pdf  →  narasimha
✅ s faiza_25.pdf  →  s faiza
✅ penabadi kavya_26.pdf  →  penabadi k

In [4]:
def extract_text(path):
    text = ""
    try:
        if path.lower().endswith(".pdf"):
            doc = fitz.open(path)
            for page in doc:
                page_text = page.get_text("text").strip()

                # If text is broken or too short, use OCR
                lines = [l.strip() for l in page_text.split("\n") if l.strip()]
                avg_len = sum(len(l) for l in lines) / max(len(lines), 1)

                if len(page_text) < 100 or avg_len < 2:
                    pix = page.get_pixmap(dpi=250)
                    img = Image.open(io.BytesIO(pix.tobytes("png")))   # ✅ now works
                    page_text = pytesseract.image_to_string(img)

                text += page_text + "\n"
            doc.close()

        elif path.lower().endswith(".docx"):
            doc = Document(path)
            for para in doc.paragraphs:
                text += para.text + "\n"

    except Exception as e:
        print(f"  ❌ {path}: {e}")

    return text.strip()

In [5]:
def clean_extracted_text(text):
    replacements = {
        "‚ĚĖ": "• ", "‚Äė": "'", "‚Äô": "'",
        "‚Äú": '"', "‚ÄĚ": '"', "‚Äď": "–", "‚ÄĒ": "—",
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # Collapse single-char-per-line garbage
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines:
        avg_len = sum(len(l) for l in lines) / len(lines)
        if avg_len < 3:
            merged, buf = [], ""
            for ch in lines:
                if len(ch) == 1:
                    if buf and ch.isupper():
                        merged.append(buf); buf = ch
                    else:
                        buf += ch
                else:
                    if buf:
                        merged.append(buf); buf = ""
                    merged.append(ch)
            if buf:
                merged.append(buf)
            text = " ".join(merged)

    text = re.sub(r"\b([a-z]) (\w)", r"\1\2", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    text = re.sub(r":\s*", ": ", text)
    text = re.sub(r"(\d) (\d)", r"\1\2", text)
    return text.strip()

In [6]:
resumes = []
seen_emails = set()

for filename, meta in name_map.items():
    path = os.path.join(CV_FOLDER, filename)

    # Skip duplicate emails
    if meta["email"] in seen_emails:
        print(f"⚠️ Duplicate email: {filename}")
        continue
    seen_emails.add(meta["email"])

    raw = extract_text(path)
    cleaned = clean_extracted_text(raw)

    if not cleaned or len(cleaned) < 20:
        print(f"⚠️ Empty: {filename}")
        continue

    resumes.append({
        "name":  meta["name"],
        "email": meta["email"],
        "text":  cleaned
    })
    print(f"  ✅ {meta['name']}: {len(cleaned)} chars")

print(f"\n✅ Total resumes: {len(resumes)}")
print(f"✅ Unique emails: {len(set(r['email'] for r in resumes))}")

  ✅ Usha.c.t: 4773 chars
  ✅ banda indu: 2298 chars
  ✅ t anil: 1733 chars
  ❌ downloaded_cvs\h pavan kalyan_5.pdf: tesseract is not installed or it's not in your PATH. See README file for more information.
  ✅ h pavan kalyan: 3518 chars
  ✅ kiran: 1788 chars
  ✅ rahul: 2196 chars
  ✅ dinesh kumar manto: 3205 chars
  ✅ mohammed akif ur rahman: 4248 chars
⚠️ Empty: Dolagobinda Mohanty_10.docx
  ✅ arpita: 2684 chars
  ✅ Deepali Dinesh B.: 3376 chars
  ✅ r ekambaram: 3573 chars
  ✅ Umarfarooq Pakali: 3881 chars
  ❌ downloaded_cvs\deepthi_15.pdf: tesseract is not installed or it's not in your PATH. See README file for more information.
⚠️ Empty: deepthi_15.pdf
  ✅ m g usharani: 2221 chars
  ✅ Sharath Chikkagoudra: 3078 chars
  ✅ Sirishala pavani: 5371 chars
  ❌ downloaded_cvs\Pallavi S_19.pdf: tesseract is not installed or it's not in your PATH. See README file for more information.
⚠️ Empty: Pallavi S_19.pdf
  ✅ Pallavi Gidnandi: 1913 chars
  ✅ Abhilash N: 2864 chars
  ✅ Sudharshan Reddy 

In [7]:
# Show first 5 to confirm name matches text
for r in resumes[:5]:
    print(f"\n{r['name']} <{r['email']}>")
    print(f"  Text: {r['text'][:100]}")


Usha.c.t <ushausha0806@gmail.com>
  Text: C T USHA
SOFTWARE DEVELOPER | DEVOPS ENGINEER
Bangalore, karnataka.
ushausha0806@gmail.com
911396736

banda indu <indubanda062@gmail.com>
  Text: RESUME
Nam e: BANDA INDU
Email Id: indubanda062@gmail.com
Mobile No: +919949384754
CAREER OBJECTIVE:

t anil <anilthuppakonda@gmail.com>
  Text: Thuppukonda Anil
#47 Hemanth nagar,Marathahalli,Bangalore
 6281003556
# anilthuppakonda@gmail.com
ï

h pavan kalyan <pavankalyan7797@gmail.com>
  Text: H PAVAN KALYAN 
• 
Mobile: +917702589818 
• 
Email: pavankalyan7797@gmail.com 
 • Linkedin: www.link

kiran <palepukiran2002@gmail.com>
  Text: PALEPU KIRAN
Andhra Pradesh | palepukiran2002@gmail.com | 9014693164 |
linkedin.com/in//palepukiran6


In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

all_documents = []

for r in resumes:
    chunks = splitter.split_text(r["text"])
    for chunk in chunks:
        if not chunk.strip():
            continue
        all_documents.append(
            LangchainDocument(
                page_content=chunk.strip(),
                metadata={"name": r["name"], "email": r["email"]}
            )
        )

print(f"✅ Total chunks: {len(all_documents)}")

✅ Total chunks: 352


In [9]:
def name_matches_text(name, text_start):
    """Check by initials + first significant word."""
    name_words = re.findall(r"\w+", name.lower())
    text_lower = text_start.lower()

    # Rule 1: Any name word >= 4 chars in text
    for w in name_words:
        if len(w) >= 4 and w in text_lower:
            return True

    # Rule 2: First initial of each name word matches start of text words
    name_initials = "".join(w[0] for w in name_words)
    text_words = re.findall(r"\w+", text_lower)[:5]
    text_initials = "".join(w[0] for w in text_words)

    return name_initials[:3] in text_initials or name_initials[:3] in text_lower

In [10]:
# Find her resume
for r in resumes:
    if r["name"] == "budamakayala rohini":
        print(r["text"][:500])
        break

Highly motivated Bachelor of Technology Computer Science student seeking an entry-level position to 
apply programming skills. Possess strong problem-solving abilities, effective collabration, and 
communication skills, eager to contribute to software development projects and grow professionally 
in adynamic work environment. 
SKILLS 
Technical Skills: Java, Python, HTML 
Behavioral Skills: Leadership, Team Collaboration, Problem Solving, Communication, Adaptability, Time 
Management 
EDUCATION 


In [11]:
# 1. Total counts
print(f"resumes:        {len(resumes)}")
print(f"all_documents:  {len(all_documents)}")
print(f"unique names:   {len(set(d.metadata['name'] for d in all_documents))}")

# 2. Every chunk has metadata
missing_meta = sum(1 for d in all_documents if not d.metadata.get("name"))
print(f"missing meta:   {missing_meta}")

# 3. No duplicate emails
emails = [r["email"] for r in resumes]
print(f"unique emails:  {len(set(emails))} / {len(emails)}")

# 4. Sample check: pick 3 random chunks and verify
import random
for doc in random.sample(all_documents, 3):
    print(f"\n--- Chunk ---")
    print(f"Meta name: {doc.metadata['name']}")
    print(f"Text start: {doc.page_content[:80]}")

resumes:        84
all_documents:  352
unique names:   84
missing meta:   0
unique emails:  84 / 84

--- Chunk ---
Meta name: b chandana sai
Text start: Containerized backend services using Docker to ensure consistent environments. 


--- Chunk ---
Meta name: h jeevan goud
Text start: Created abasic Amazon Sales Dashboard in Power BI to visualize sales and order d

--- Chunk ---
Meta name: venkata prasanth
Text start: • Foundation – Physical Design (maveen silicon).
• Cyber Security
ADDITIONAL INF


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print(f"🔨 Building FAISS index from {len(all_documents)} chunks...")

vectorstore = FAISS.from_documents(all_documents, model)
vectorstore.save_local("faiss_cv_index")

print(f"✅ FAISS index: {vectorstore.index.ntotal} vectors")
print(f"✅ Saved to ./faiss_cv_index/")

Loading weights: 100%|██████████| 199/199 [00:02<00:00, 84.49it/s]


🔨 Building FAISS index from 352 chunks...
✅ FAISS index: 352 vectors
✅ Saved to ./faiss_cv_index/


In [13]:
from openai import OpenAI
from collections import defaultdict
import json, time, re

client = OpenAI(
    api_key="AQ.Ab8RN6LKp8hrYJZLcAv_2SQXREpNDN1ddX7J71XZlDgdp-ZUJw",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [14]:
jd = """IIIT Allahabad, Uttar Pradesh
Duration: 20 days offline
Global Academy of Technology, Bangalore
CGPA: 7.7
Pre University in PCMB 2018 - 2022
Sri Bhagavan Mahaveer Jain College, Bangalore
Percentage: 84
1
AI & Data Science Fresher | +91-8147558661 | 11nikitam11@gmail.com
https://www.linkedin.com/in/nikita-m-73a93b271/ | https://github.com/nike-xyz
Programming: Python (NumPy, Pandas, Matplotlib, Scikit-learn, Seaborn, OOPs), SQL, HTML, CSS, JS, Django.
Data & AI: Machine Learning, Data Cleaning, Prompt Engineering, Model Training and Testing.
Tools: Jupyter Notebook, Google Colab, VS Code, Power BI, Excel, Tableau.
Marketing tech: Facebook ads manager (Meta ads), Google Analytics, Campaign performance analysis.
Soft skills: Problem-solving, Analytical thinking, client communication, data-driven decision making.
Projects
Objective: Grocery item detection using computer vision.
Steps followed: Data collection, data annotation, data augmentation, loaded data to jupyter notebook, split the data
into train & test, trained the model, tested the model, evaluation, parameters tuning.
Language: Python. Editor: Google Colab. Packages used: ultralytics, roboflow.
Models used: Yolo V8n, CNN, pytesseract, Azure cloud OCR model, Google cloud OCR model.
Conclusion: Yolo V8n performed the best with highest accuracy.
Results-driven sales professional with hands-on experience in Meta Ads analytics and a B.Tech in Artificial
Intelligence & Data Science (2024), now transitioning into a full-time AI and Data Science role. Completed an AI,
Data Science, and Python Full Stack programme at Palle Technologies to strengthen practical and industry-ready
skills. Eager to combine a strong academic foundation, real-world campaign analytics experience, and emerging
technical expertise to contribute to impactful AI and data projects.
Technical Skills
Objective: Predict whether a patient has diabetes.
Steps followed: Used Pima dataset, loaded data to jupyter notebook, cleaned the data, split into train/test sets, trained
the model, tested the model, evaluated & improved performance.
Language: Python. Editor: jupyter notebook. Packages used: NumPy, Pandas, Matplotlib, Scikit-learn, Seaborn.
Models used: Naive bayes, Random forest, KNN.
Conclusion: Random forest performed the best with highest accuracy, precision, recall & f1 score.
Objective: Detecting whether an object passed a sensor.
Steps followed: Integrating an ultrasonic sensor with Rasberry pi, Received data from ultrasonic sensors; plotted realtime data dynamically using Tkinter.
Language: Python, terminal commands, Arduino. Editor: Thonny. Hardware used: Arduino, ultrasonic sensor, Rasberry
pi
Education
Summary
Internship
Multi media analysis
Nikita M
AI smart cart 2024
2022
B.Tech (AI and Data Science)
IOT real time streaming data analysis
2020 - 2024
Diabetes Prediction with Machine Learning
2023
2026"""

In [15]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 60, "fetch_k": 120}
)
context = retriever.invoke(jd)

In [16]:
print(type(context))
print(len(context))
for i in context:
    print()
    print(i)
    print()
    print('=='*60)
    print()


<class 'list'>
60

page_content='Marketing tech: Facebook ads manager (Meta ads), Google Analytics, Campaign performance analysis.
Soft skills: Problem-solving, Analytical thinking, client communication, data-driven decision making.
Projects
Objective: Grocery item detection using computer vision.
Steps followed: Data collection, data annotation, data augmentation, loaded data to jupyter notebook, split the data
into train & test, trained the model, tested the model, evaluation, parameters tuning.
Language: Python. Editor: Google Colab. Packages used: ultralytics, roboflow. 
Models used: Yolo V8n, CNN, pytesseract, Azure cloud OCR model, Google cloud OCR model.
Conclusion: Yolo V8n performed the best with highest accuracy.' metadata={'name': 'nikita m', 'email': '11nikitam11@gmail.com'}



page_content='IIIT Allahabad, Uttar Pradesh
Duration: 20 days offline
 
Global Academy of Technology, Bangalore
CGPA: 7.7
Pre University in PCMB 2018 - 2022
Sri Bhagavan Mahaveer Jain College, Bangal

In [17]:

jd = """
Company: Quantacus
Job Role: AI Implementation Engineer (2 openings)
Location: Bangalore

We are looking for highly analytical, SQL-first engineers.
This is NOT a dashboarding role.

What we care about:
- Strong SQL skills
- Analytical reasoning
- Curiosity toward AI and automation
- 0-2 Yrs experience. Strong freshers encouraged.

NOT suitable for:
- Power BI dashboard development
- Tableau dashboarding
"""

In [18]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 60, "fetch_k": 120}
)
context = retriever.invoke(jd)

In [19]:
grouped = defaultdict(lambda: {"name": "", "email": "", "text": ""})
for doc in context:
    name  = doc.metadata.get("name", "Unknown")
    email = doc.metadata.get("email", "")
    grouped[name]["name"]  = name
    grouped[name]["email"] = email
    grouped[name]["text"] += doc.page_content + "\n\n"

# Take the top 15 unique candidates (by first-seen order = relevance order)
candidates = list(grouped.values())[:15]

print(f"✅ Top 15 candidates selected:")
for i, c in enumerate(candidates, 1):
    print(f"  {i}. {c['name']}  ({len(c['text'])} chars)")

✅ Top 15 candidates selected:
  1. pilli veena  (779 chars)
  2. neelakanta  (2864 chars)
  3. apoorva s goranal  (1510 chars)
  4. Umarfarooq Pakali  (1089 chars)
  5. abdul subhan  (783 chars)
  6. sravanthi  (786 chars)
  7. anant kumar  (695 chars)
  8. t anil  (1544 chars)
  9. rahul p t  (1506 chars)
  10. sravani  (351 chars)
  11. Soundarya B H  (765 chars)
  12. lakshminath  (1137 chars)
  13. Pallavi Gidnandi  (525 chars)
  14. b chandana sai  (2285 chars)
  15. t chandrika  (417 chars)


In [20]:
# Score
results = []

for i, cand in enumerate(candidates, 1):
    print(f"[{i}/{len(candidates)}] {cand['name']}")

    prompt = f"""
<job>
{jd}
</job>

<resume>
{cand['text']}
</resume>

Return ONLY JSON: {{"score": <int>}}
"""

    score = 0
    for attempt in range(5):
        try:
            response = client.chat.completions.create(
                model="gemini-3.5-flash-lite",   # 15 req/min
                messages=[
                    {"role": "system", "content": "Output only valid JSON."},
                    {"role": "user", "content": prompt}
                ]
            )
            raw = response.choices[0].message.content.replace("```json","").replace("```","").strip()
            score = json.loads(raw).get("score", 0)
            print(f"   ✅ {score}")
            break
        except Exception as e:
            err = str(e)
            if "429" in err or "RESOURCE_EXHAUSTED" in err:
                m = re.search(r"retry in ([\d.]+)s", err)
                wait = float(m.group(1)) + 3 if m else 10
                print(f"   ⏳ {wait:.0f}s")
                time.sleep(wait)
            else:
                print(f"   ❌ {err[:80]}")
                time.sleep(3)

    results.append({"name": cand["name"], "email": cand["email"], "score": score})
    time.sleep(5)

[1/15] pilli veena
   ✅ 75
[2/15] neelakanta
   ✅ 25
[3/15] apoorva s goranal
   ✅ 75
[4/15] Umarfarooq Pakali
   ✅ 85
[5/15] abdul subhan
   ✅ 50
[6/15] sravanthi
   ✅ 10
[7/15] anant kumar
   ✅ 85
[8/15] t anil
   ✅ 65
[9/15] rahul p t
   ✅ 50
[10/15] sravani
   ✅ 25
[11/15] Soundarya B H
   ✅ 50
[12/15] lakshminath
   ✅ 75
[13/15] Pallavi Gidnandi
   ✅ 20
[14/15] b chandana sai
   ✅ 40
[15/15] t chandrika
   ✅ 40


In [21]:
results.sort(key=lambda x: x["score"], reverse=True)

print("\n" + "=" * 60)
print("🏆 FINAL ATS RANKING")
print("=" * 60)
for i, r in enumerate(results, 1):
    print(f"#{i}  {r['name']}  —  {r['score']}/100  ({r['email']})")


🏆 FINAL ATS RANKING
#1  Umarfarooq Pakali  —  85/100  (Umarfarukhpakali1234@gmail.com)
#2  anant kumar  —  85/100  (anantkumar220120@gmail.com)
#3  pilli veena  —  75/100  (pilliveenanvr@gmail.com)
#4  apoorva s goranal  —  75/100  (apoorvasg2004@gmail.com)
#5  lakshminath  —  75/100  (pallalakshminath@gmail.com)
#6  t anil  —  65/100  (anilthuppakonda@gmail.com)
#7  abdul subhan  —  50/100  (abdlsubhan05@gmail.com)
#8  rahul p t  —  50/100  (rahultopugol@gmail.com)
#9  Soundarya B H  —  50/100  (soundaryabh096@gmail.com)
#10  b chandana sai  —  40/100  (chandanabollini@gmail.com)
#11  t chandrika  —  40/100  (tallamchandrika006@gmail.com)
#12  neelakanta  —  25/100  (neelakanta2853@gmail.com)
#13  sravani  —  25/100  (sravanisravani6165@gmail.com)
#14  Pallavi Gidnandi  —  20/100  (pallusg2004@gmail.com)
#15  sravanthi  —  10/100  (yaddalapudisravanthi@gmail.com)
